In [2]:
import sympy as sp
import sympy.diffgeom as dg

In [3]:
state_mfld = dg.Manifold("M", 5)
state_mfld_patch = dg.Patch("P", state_mfld)

x, y, theta, v, omega = sp.symbols(r"x,y,\theta,v,\omega", real=True)

state_mfld_coords = dg.CoordSystem(
    "StateSpace", state_mfld_patch, (x, y, theta, v, omega)
)

(x_sc, y_sc, theta_sc, v_sc, omega_sc) = state_mfld_coords.base_scalars()
(x_vec, y_vec, theta_vec, v_vec, omega_vec) = state_mfld_coords.base_vectors()

In [17]:
# define the vector fields of the system

f = (
    v_sc * sp.cos(theta_sc) * x_vec
    + v_sc * sp.sin(theta_sc) * y_vec
    + omega_sc * theta_vec
)

f1 = v_vec
f2 = omega_vec

u_f, u_t = sp.symbols("u_f,u_t", real=True)

display(f, f1, f2)
display(u_f, u_t)

sin(\theta)*v*e_y + cos(\theta)*v*e_x + \omega*e_\theta

e_v

e_\omega

u_f

u_t

In [18]:
# define the variables associated with the keep out region
t = sp.symbols("t", real=True)

x_ko = sp.Function("x_ko")(t)
y_ko = sp.Function("y_ko")(t)
r_ko = sp.Function("r_ko")(t)

ko_cbf = sp.sqrt((x_sc - x_ko) ** 2 + (y_sc - y_ko) ** 2) - r_ko
display(ko_cbf)

sqrt((-x_ko(t) + x)**2 + (-y_ko(t) + y)**2) - r_ko(t)

In [45]:
# generates the condition for the HOCBF


def _repeated_lie_deriv(f, h, n):
    if n == 0:
        return h
    return _repeated_lie_deriv(f, dg.LieDerivative(f, h), n - 1)


def gen_hocbf_condition(dyn_f, dyn_gs, us, m, h, t, alpha_fns):
    lie_m_fh = _repeated_lie_deriv(dyn_f, h, m)
    lie_g_fh = sum(
        dg.LieDerivative(g, _repeated_lie_deriv(f, h, m - 1)) * u
        for (g, u) in zip(dyn_gs, us)
    )
    dmh_dtm = sp.diff(h, t, m)

    alpha_fns.insert(0, lambda phi: 1.0 * phi)  # dummy to ensure correct indexing

    phi_fns = [h]
    for i in range(1, m):
        phi_fns.append(sp.diff(phi_fns[i - 1], t) + alpha_fns[i](phi_fns[i - 1]))

    o_h = 0
    for i in range(1, m - 1):
        # same thing here with the alpha indexing
        alpha_phi_fn = alpha_fns[m - i](phi_fns[m - i - 1])
        o_h += _repeated_lie_deriv(f, alpha_phi_fn, i) + sp.diff(alpha_phi_fn, t, i)

    final_alpha_phi = alpha_fns[m](phi_fns[m - 1])

    return lie_m_fh + lie_g_fh + dmh_dtm + o_h + final_alpha_phi


# assuming the use of linear extended K_infty functions
k1, k2 = sp.symbols("k1,k2", nonneg=True, real=True)

ko_hocbf = (
    gen_hocbf_condition(
        f, [f1, f2], [u_f, u_t], 2, ko_cbf, t, [lambda x: k1 * x, lambda x: k2 * x]
    )
    .nsimplify()
    .simplify()
)

display(ko_hocbf)  # ouch

(-((((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(x_ko(t) - x) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*cos(\theta))*cos(\theta) + (((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(y_ko(t) - y) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*sin(\theta))*sin(\theta))*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2)*v**2 - ((x_ko(t) - x)*Derivative(x_ko(t), t) + (y_ko(t) - y)*Derivative(y_ko(t), t))**2*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2) + ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(7/2)*(-k2*((-k1*(sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - r_ko(t)) + Derivative(r_ko(t), t))*sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - (x_ko(t) - x)*Derivative(x_ko(t), t) - (y_ko(t) - y)*Derivative(y_ko(t), t)) - u_f*((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta)) + ((x_ko(t) - x)*sin(\theta) + (-y_ko(t) + y)*cos(\theta))*v*\omega + (x_ko(t) - x)*Derivative(x_ko(t), (t, 2)) + (y_ko(t) - y)*Derivative(y_ko(t), (t, 2)) + Derivative(x_ko(t), t)**2 + Derivative(y_ko(t), t)**2) - ((x_ko(t

In [72]:
# devise some simplifications that might help

x_err_sym, y_err_sym, v_err_sym, r_1_sym, r_2_sym, r_3_sym, r_4_sym, r_5_sym = (
    sp.symbols("x_err,y_err,v_err,r_1,r_2,r_3,r_4,r_5", real=True)
)

x_err = x_ko - x_sc
y_err = y_ko - y_sc
v_err = x_err_sym**2 + y_err_sym**2

r_1 = v_err_sym ** (sp.Rational(7 / 2)) * x_err_sym
r_2 = v_err_sym ** (sp.Rational(7 / 2)) * y_err_sym
r_3 = v_err_sym ** (sp.Rational(5 / 2)) * x_err_sym**2
r_4 = v_err_sym ** (sp.Rational(5 / 2)) * y_err_sym**2
r_5 = v_err_sym ** (sp.Rational(5 / 2)) * x_err_sym * y_err_sym

ko_hocbf_cleaned = (
    (ko_hocbf.subs(x_err, x_err_sym).subs(y_err, y_err_sym).subs(v_err, v_err_sym))
    .simplify()
    .factor()
    .subs(r_1, r_1_sym)
    .subs(r_2, r_2_sym)
    .subs(r_3, r_3_sym)
    .subs(r_4, r_4_sym)
    .subs(r_5, r_5_sym)
).factor()
display(ko_hocbf_cleaned)

-(-2*k1*k2*v_err**(9/2) + 2*k1*k2*v_err**4*r_ko(t) - 2*k2*r_1*Derivative(x_ko(t), t) - 2*k2*r_2*Derivative(y_ko(t), t) + 2*k2*v_err**4*Derivative(r_ko(t), t) + 2*r_1*u_f*cos(\theta) - 2*r_1*sin(\theta)*v*\omega - 2*r_1*Derivative(x_ko(t), (t, 2)) + 2*r_2*u_f*sin(\theta) + 2*r_2*cos(\theta)*v*\omega - 2*r_2*Derivative(y_ko(t), (t, 2)) + r_3*cos(2*\theta)*v**2 + r_3*v**2 + 2*r_3*Derivative(x_ko(t), t)**2 - r_4*cos(2*\theta)*v**2 + r_4*v**2 + 2*r_4*Derivative(y_ko(t), t)**2 + 2*r_5*sin(2*\theta)*v**2 + 4*r_5*Derivative(x_ko(t), t)*Derivative(y_ko(t), t) - 2*v_err**(7/2)*v**2 - 2*v_err**(7/2)*Derivative(x_ko(t), t)**2 - 2*v_err**(7/2)*Derivative(y_ko(t), t)**2 + 2*v_err**4*Derivative(r_ko(t), (t, 2)))/(2*v_err**4)

In [87]:
# final HOCBF (to use as condition in optimization problem)
ko_hocbf_condition = ko_hocbf_cleaned >= 0
display(ko_hocbf_condition)

display(sp.Eq(x_err_sym, x_err))
display(sp.Eq(y_err_sym, y_err))
display(sp.Eq(v_err_sym, v_err))
display(sp.Eq(r_1_sym, r_1))
display(sp.Eq(r_2_sym, r_2))
display(sp.Eq(r_3_sym, r_3))
display(sp.Eq(r_4_sym, r_4))
display(sp.Eq(r_5_sym, r_5))

sp.print_latex(ko_hocbf_condition)

-(-2*k1*k2*v_err**(9/2) + 2*k1*k2*v_err**4*r_ko(t) - 2*k2*r_1*Derivative(x_ko(t), t) - 2*k2*r_2*Derivative(y_ko(t), t) + 2*k2*v_err**4*Derivative(r_ko(t), t) + 2*r_1*u_f*cos(\theta) - 2*r_1*sin(\theta)*v*\omega - 2*r_1*Derivative(x_ko(t), (t, 2)) + 2*r_2*u_f*sin(\theta) + 2*r_2*cos(\theta)*v*\omega - 2*r_2*Derivative(y_ko(t), (t, 2)) + r_3*cos(2*\theta)*v**2 + r_3*v**2 + 2*r_3*Derivative(x_ko(t), t)**2 - r_4*cos(2*\theta)*v**2 + r_4*v**2 + 2*r_4*Derivative(y_ko(t), t)**2 + 2*r_5*sin(2*\theta)*v**2 + 4*r_5*Derivative(x_ko(t), t)*Derivative(y_ko(t), t) - 2*v_err**(7/2)*v**2 - 2*v_err**(7/2)*Derivative(x_ko(t), t)**2 - 2*v_err**(7/2)*Derivative(y_ko(t), t)**2 + 2*v_err**4*Derivative(r_ko(t), (t, 2)))/(2*v_err**4) >= 0

Eq(x_err, x_ko(t) - x)

Eq(y_err, y_ko(t) - y)

Eq(v_err, x_err**2 + y_err**2)

Eq(r_1, v_err**(7/2)*x_err)

Eq(r_2, v_err**(7/2)*y_err)

Eq(r_3, v_err**(5/2)*x_err**2)

Eq(r_4, v_err**(5/2)*y_err**2)

Eq(r_5, v_err**(5/2)*x_err*y_err)

- \frac{- 2 k_{1} k_{2} v_{err}^{\frac{9}{2}} + 2 k_{1} k_{2} v_{err}^{4} r_{ko}{\left(t \right)} - 2 k_{2} r_{1} \frac{d}{d t} x_{ko}{\left(t \right)} - 2 k_{2} r_{2} \frac{d}{d t} y_{ko}{\left(t \right)} + 2 k_{2} v_{err}^{4} \frac{d}{d t} r_{ko}{\left(t \right)} + 2 r_{1} u_{f} \cos{\left(\mathbf{\theta} \right)} - 2 r_{1} \sin{\left(\mathbf{\theta} \right)} \mathbf{v} \mathbf{\omega} - 2 r_{1} \frac{d^{2}}{d t^{2}} x_{ko}{\left(t \right)} + 2 r_{2} u_{f} \sin{\left(\mathbf{\theta} \right)} + 2 r_{2} \cos{\left(\mathbf{\theta} \right)} \mathbf{v} \mathbf{\omega} - 2 r_{2} \frac{d^{2}}{d t^{2}} y_{ko}{\left(t \right)} + r_{3} \cos{\left(2 \mathbf{\theta} \right)} \mathbf{v}^{2} + r_{3} \mathbf{v}^{2} + 2 r_{3} \left(\frac{d}{d t} x_{ko}{\left(t \right)}\right)^{2} - r_{4} \cos{\left(2 \mathbf{\theta} \right)} \mathbf{v}^{2} + r_{4} \mathbf{v}^{2} + 2 r_{4} \left(\frac{d}{d t} y_{ko}{\left(t \right)}\right)^{2} + 2 r_{5} \sin{\left(2 \mathbf{\theta} \right)} \mathbf{v}^{2} + 4 r_{5} \